# 전처리

핵심적인 전처리 기술 5가지

1. 타겟 레이블링: Piecewise RUL (조각별 잔여 수명)
도메인 지식 - 엔진의 초기 상태는 매우 건강하기 때문에 초기 사이클에서는 RUL이 줄어드는 것이 의미가 없음
원리: 실제 수명이 300 사이클이 남았더라도 특정 임계값(예: 125)으로 캡핑(Capping)함
효과: 초기 정상 작동 구간에서의 불필요한 노이즈 학습을 방지하고, 고장이 임박한 시점의 변화에 모델이 더 집중

2. 시계열 피처 엔지니어링 (Feature Engineering)
단순한 현재 시점의 센서값뿐만 아니라 시간의 흐름을 반영하는 변수들을 생성
Cycle Normalization: cycle / max_cycle을 통해 엔진마다 다른 전체 수명을 0~1 사이의 상대적 진행률로 변환하여 모델이 엔진의 마모 정도를 쉽게 파악
Smoothing (이동 평균): rolling(window=5).mean()을 사용해 센서 데이터 특유의 뾰족한 노이즈를 제거하여 추세를 부드럽게 만듬
Differencing (차분): 이전 시점 대비 센서값의 변화량(diff)을 계산하여 수치가 급격히 변하는 '열화 징후'를 포착

3. 작동 조건별 클러스터링 (K-Means Clustering)
FD002, FD004 데이터셋처럼 비행 고도나 속도가 제각각인 경우에 필수적인 스킬
원리: 3개의 작동 설정값(op1, op2, op3)을 기준으로 엔진이 현재 어떤 환경(예: 고고도 비행 중, 이륙 중 등)에 있는지 6개의 클러스터로 분류
효과: 환경에 따라 센서의 기준값이 달라지는 문제를 해결하기 위해, 각 환경군 내에서 개별적인 통계 처리

4. 그룹 기반 스케일링 및 Fallback 전략
데이터의 분포를 맞추는 스케일링 단계에서 발생할 수 있는 '데이터 누수(Leakage)'와 '예외 상황'을 동시에 잡는 고급 기법
Condition-wise Scaling: 전체 데이터를 한꺼번에 정규화하지 않고, 앞서 나눈 클러스터(작동 조건)별로 각각 StandardScaler를 적용합니다. 이는 각 조건에서의 센서 편차를 표준화하여 모델이 조건에 관계없이 일관된 패턴을 읽음
Global Fallback: 테스트 데이터나 검증 데이터에 학습 시 보지 못한 조건이 등장할 경우를 대비해 전체 평균으로 정규화하는 global_scaler를 준비하여 코드의 안정성을 높임

5. 동적 피처 선택 (LGBM Importance 기반)
모든 피처를 다 쓰는 대신, 모델 학습에 방해가 되는 불필요한 변수를 걸러내는 과정
Inductive Bias 제어: 사람이 임의로 피처를 고르는 대신, 가벼운 LightGBM 모델을 먼저 돌려본 뒤 중요도가 높은 상위 12개 피처만 선택
Must-have Features: 중요도가 낮더라도 도메인상 핵심적인 변수(OP_COLS, cycle_norm)는 무조건 포함되도록 로직을 짜서 모델의 최소 성능을 보장

요약하자면
이 코드는 "노이즈 제거(Smoothing) → 환경 분류(Clustering) → 타겟 보정(Piecewise) → 조건별 정규화(Scaling) → 핵심 변수 추출(Selection)"이라는 탄탄한 시계열 분석 파이프라인을 갖추고 있음(특히 작동 조건별로 스케일러를 따로 관리)

In [2]:
# 1. 라이브러리 임포트 및 초기 설정
import os
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import GroupKFold  # 엔진 유닛(unit)별로 겹치지 않게 교차 검증하기 위함
from sklearn.preprocessing import StandardScaler # 데이터 정규화를 위한 스케일러
from sklearn.metrics import mean_squared_error # 평가 지표(RMSE) 계산
from sklearn.cluster import KMeans             # 작동 조건(Operating Conditions) 분류를 위한 클러스터링
from lightgbm import LGBMRegressor             # 메인 예측 모델

warnings.filterwarnings("ignore") # 불필요한 경고 메시지 출력 방지

# 데이터셋의 컬럼 이름 정의 (엔진번호, 사이클, 3개의 설정값, 21개의 센서값)
COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
OP_COLS = ["op1", "op2", "op3"] # 작동 설정 관련 컬럼

ModuleNotFoundError: No module named 'lightgbm'

In [ ]:
# 2. 데이터 로드 및 전처리 함수
def load_cmapss(dataset: str, data_dir: str = "CMAPSSData"):
    # 공백으로 구분된 텍스트 파일을 읽어오기 위한 설정
    kw = dict(sep=r"\s+", header=None, names=COLS, engine="python")
    # 학습 데이터, 테스트 데이터, 실제 RUL 정답 값을 로드
    train = pd.read_csv(os.path.join(data_dir, f"train_{dataset}.txt"), **kw)
    test  = pd.read_csv(os.path.join(data_dir, f"test_{dataset}.txt"),  **kw)
    rul   = pd.read_csv(os.path.join(data_dir, f"RUL_{dataset}.txt"),
                        sep=r"\s+", header=None, names=["RUL"], engine="python")
    return train, test, rul

def add_advanced_features(df: pd.DataFrame, sensors: list):
    df = df.sort_values(['unit', 'cycle']) # 엔진별, 시간순 정렬
    # 현재 사이클이 전체 수명 중 어느 정도 위치인지 0~1 사이로 정규화
    df["cycle_norm"] = df.groupby("unit")["cycle"].transform(lambda x: x / x.max())

    grouped = df.groupby("unit")
    for s in sensors:
        # 노이즈 제거를 위해 윈도우 크기 5의 이동 평균(Moving Average) 계산
        s_smooth = grouped[s].transform(
            lambda x: x.rolling(window=5, min_periods=5).mean()
        ).fillna(method="bfill").fillna(method="ffill") # 결측치는 앞뒤 값으로 채움
        
        # 센서 데이터의 변화량(직전 대비 차이) 피처 생성
        df[f"{s}_diff"] = s_smooth.groupby(df["unit"]).diff().fillna(0)
        df[s] = s_smooth # 원본 센서값을 매끄럽게 처리된 값으로 교체
    return df.reset_index(drop=True) #추가

def apply_piecewise_rul(df: pd.DataFrame, dataset: str):
    # 초기 상태의 엔진은 RUL을 일정하게 유지시키는 Piecewise RUL 기법 적용 (성능 향상에 필수적)
    cap = 135 if "FD002" in dataset or "FD004" in dataset else 125
# 이건........ 변경이 필요...
    mc = df.groupby("unit")["cycle"].max().rename("max_cycle") # 엔진별 최대 사이클 계산
    df = df.join(mc, on="unit")
    # (최대 사이클 - 현재 사이클)을 계산하되 설정한 cap(상한선)을 넘지 않도록 제한
    df["RUL"] = (df["max_cycle"] - df["cycle"]).clip(upper=cap)
    return df.drop("max_cycle", axis=1)

In [ ]:
# 3. 피처 선택 (Feature Selection)
def select_features_fold(df, candidate_features):
    """모델의 중요도를 기반으로 유연하게 피처를 선택하는 함수"""
    model = LGBMRegressor(n_estimators=300, random_state=42)
    model.fit(df[candidate_features], df["RUL"]) # 임시 모델 학습
    
    # 각 피처의 중요도를 계산하여 내림차순 정렬
    importance = pd.Series(model.feature_importances_, index=candidate_features).sort_values(ascending=False)
    
    # 반드시 포함해야 할 피처(작동 조건 및 정규화된 사이클) 정의
    must = OP_COLS + ["cycle_norm"]
    others = [f for f in importance.index if f not in must]
    # 필수 피처를 포함하여 중요도 상위 총 12개의 피처를 최종 선택
    final = must + others[:(12 - len(must))]
    
    return final

In [ ]:
# 4. 메인 실험 루프 (학습 및 평가)
def run_experiment(dataset: str):
    # 데이터 로드 및 전처리
    train_raw, test_raw, rul_df = load_cmapss(dataset)
    # 표준편차가 너무 낮은(변화가 거의 없는) 센서는 제외
    stds = train_raw[[f"s{i}" for i in range(1, 22)]].std()
    sensors = stds[stds > 0.01].index.tolist()
    
    # 피처 엔지니어링 및 RUL 타겟 생성
    train_df = add_advanced_features(train_raw, sensors)
    test_df = add_advanced_features(test_raw, sensors)
    train_df = apply_piecewise_rul(train_df, dataset)
    
    # 모델 학습에 사용할 후보 피처 리스트업
    candidate_features = OP_COLS + sensors + ["cycle_norm"] + [f"{s}_diff" for s in sensors]
    
    # 작동 조건이 복잡한 FD002/004 데이터셋은 K-Means로 조건을 6개 그룹으로 클러스터링
    if dataset in ["FD002", "FD004"]:
        km = KMeans(n_clusters=6, random_state=42)
        train_df["_cond"] = km.fit_predict(train_df[OP_COLS])
        test_df["_cond"] = km.predict(test_df[OP_COLS])
    else:
        train_df["_cond"] = 0 # 단일 조건 데이터셋은 그룹을 0으로 통일
        test_df["_cond"] = 0

    # 엔진 유닛(unit)이 섞이지 않게 5-Fold 교차 검증 설정
    gkf = GroupKFold(n_splits=5)
    oof_preds = np.zeros(len(train_df)) # Out-of-Fold 예측값 저장용
    models_info = [] # 각 폴드에서 학습된 모델과 스케일러 저장용

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(train_df, train_df["RUL"], train_df["unit"])):
        tr_fold = train_df.iloc[tr_idx].copy()
        val_fold = train_df.iloc[val_idx].copy()

        # 현재 폴드 데이터에 최적화된 상위 피처 선택
        selected_features = select_features_fold(tr_fold, candidate_features)
        
        # 전체 학습 데이터에 대한 글로벌 스케일러 생성 (Fallback용)
        global_scaler = StandardScaler()
        global_scaler.fit(tr_fold[selected_features])
        
        scalers_per_fold = {}
        all_conds = sorted(set(tr_fold["_cond"]) | set(val_fold["_cond"]))

        for c in all_conds:
            c_tr_idx = tr_fold["_cond"] == c
            c_val_idx = val_fold["_cond"] == c
            
            if c_tr_idx.any():
                # 같은 작동 조건 그룹끼리 따로 스케일링 수행 (정규화의 핵심)
                scaler = StandardScaler()
                scaler.fit(tr_fold.loc[c_tr_idx, selected_features])
                tr_fold.loc[c_tr_idx, selected_features] = scaler.transform(tr_fold.loc[c_tr_idx, selected_features])
                
                if c_val_idx.any():
                    val_fold.loc[c_val_idx, selected_features] = scaler.transform(val_fold.loc[c_val_idx, selected_features])
                
                scalers_per_fold[c] = scaler
            else:
                # 검증셋에는 있지만 학습셋에는 없는 조건일 경우, 글로벌 스케일러로 대체하여 에러 방지
                if c_val_idx.any():
                    val_fold.loc[c_val_idx, selected_features] = global_scaler.transform(val_fold.loc[c_val_idx, selected_features])

        # LightGBM 모델 정의 및 학습 (Early Stopping 적용)
        model = LGBMRegressor(n_estimators=1000, learning_rate=0.03, num_leaves=31, random_state=42)
        model.fit(tr_fold[selected_features], tr_fold["RUL"], 
                  eval_set=[(val_fold[selected_features], val_fold["RUL"])],
                  eval_metric='rmse', early_stopping_rounds=50, verbose=False)

        # 검증 데이터 예측값 저장 및 모델 정보 보관
        oof_preds[val_idx] = model.predict(val_fold[selected_features])
        models_info.append((model, selected_features, scalers_per_fold, global_scaler))
        print(f"Fold {fold+1} 완료")

    # 전체 학습 데이터에 대한 RMSE 출력
    print(f"\n🚀 [{dataset}] OOF RMSE: {np.sqrt(mean_squared_error(train_df['RUL'], oof_preds)):.4f}")

    # 5. 테스트 데이터 예측
    # 각 엔진별 가장 마지막 시점(고장 직전)의 데이터만 추출
    test_last_idx = test_df.groupby("unit").tail(1).index
    y_test_raw = rul_df["RUL"].values
    test_preds_list = []

    for model, features, scalers, g_scaler in models_info:
        temp_test = test_df.copy()
        
        # 테스트 데이터도 학습 시와 동일하게 작동 조건별로 스케일링 적용
        for c in temp_test["_cond"].unique():
            te_c_idx = temp_test["_cond"] == c
            if te_c_idx.any():
                # 해당 조건의 스케일러가 없으면 글로벌 스케일러 사용(Fallback)
                scaler = scalers.get(c, g_scaler)
                temp_test.loc[te_c_idx, features] = scaler.transform(temp_test.loc[te_c_idx, features])
        
        # 각 폴드 모델의 예측 수행
        p = model.predict(temp_test.loc[test_last_idx, features])
        test_preds_list.append(p)
    
    # 5개 폴드 모델의 예측값을 평균 내어 최종 앙상블 예측값 계산
    final_test_preds = np.mean(test_preds_list, axis=0)
    print(f"📊 [{dataset}] 최종 TEST RMSE: {np.sqrt(mean_squared_error(y_test_raw, final_test_preds)):.4f}")

    return final_test_preds